In [2]:
import numpy as np
import pandas as pd

### **Создадим данные.**

In [4]:
#признак от, которого по-настоящему зависит Y
#uniform — это равномерное распределение случайных чисел. (от, до, сколько)
n = 300 #кол-во признаков
x = np.random.uniform(1, 100, n)

_Создадим датафрейм._

In [5]:
#создадим пустую таблицу
X = pd.DataFrame()
#укажем в ней столбик изначальных признаков
X['f0'] = x

_Добавим данные с сильной мультиколлинеарностью. Новые признаки будут почти полностью равны изначальному._

In [6]:
for i in range(1, 20):
    noise = np.random.normal(0, 0.01, n)
    new_f = x + noise
    X[f'f{i}'] = new_f

_Создадим вектор ответов._

In [7]:
noise = np.random.normal(0, 20, n)
Y = 5 * x + noise

### **Поделим данные на тренировочные и тестовые.**

In [9]:
from sklearn.model_selection import train_test_split
#получим разделенные данные
x_tr, x_ts, y_tr, y_ts = train_test_split(X, Y,
                                          test_size=0.3,
                                          random_state=33)

### **Напишем функцию для реализации одного шага градиентного спуска линейной регрессии.**

In [18]:
def grad(X, Y, w, lr):
    X = X.to_numpy()
    #вычислим текущее предсказание
    pred = X @ w #вектор
    #найдем ошибку на каждом объекте
    error = pred - Y #вектор
    #вычислим вклад каждого объекта в градиент по каждому весу
    #ошибку на этом объекте распространить на все признаки
    grad = 2 * X * error[:, None] #матрица
    #средний градиент для каждого веса
    grad_mean = np.mean(grad, axis=0) #вектор
    #сделаем шаг в обратную сторону
    w = w - grad_mean * lr
    return w

### **Напишем свою функцию fit для линейной регрессии.**

In [28]:
def fit(X, Y):
    #создадим изначальные веса
    w = np.zeros(X.shape[1])
    #разница предыдущей и текущей ошибки
    dif = 10
    while dif > 1e-6:
        #предыдущая ошибка
        last_error = np.mean((X @ w - Y) ** 2)
        #сделаем один шаг спуска
        w = grad(X, Y, w, 1e-5)
        #узнаем текущую ошибку
        cur_error = np.mean((X @ w - Y) ** 2)
        #обновим разницу ошибки
        dif = abs(last_error - cur_error)
    return w

_Обучим модель._

In [29]:
new_w = fit(x_tr, y_tr)
#узнаем её ошибку
pred = (x_ts @ new_w)
error = np.mean((pred - y_ts)**2)
error

np.float64(388.01972228504513)

_Посмотрим на ей коэффициенты._

In [30]:
new_w

array([0.25047126, 0.25047421, 0.25047523, 0.25047111, 0.25047013,
       0.25047306, 0.25047295, 0.25046732, 0.25046853, 0.25047198,
       0.25047497, 0.2504734 , 0.25047164, 0.25046992, 0.25047325,
       0.2504793 , 0.25046941, 0.25047062, 0.2504709 , 0.2504755 ])

### **Проведем стандартизацию признаков.**

_Вычислим стандартное отклонение._

In [31]:
#вычислим среднее каждого признака
X_mean = np.mean(X, axis=0) #вектор
#вычислим квадрат отклонения каждого значения от среднего
X_squared_diff = (X - X_mean)**2 #матрица
#найдем среднее квадратов отклонений
X_var = np.mean(X_squared_diff, axis=0) #вектор
#найдем стандартное отклонение
std = np.sqrt(X_var) #вектор

_Стандартизируем данные._

In [32]:
X_std = (X - X_mean) / std
X_std

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19
0,-0.284003,-0.284015,-0.283907,-0.283338,-0.283983,-0.283620,-0.284025,-0.283639,-0.284025,-0.283755,-0.284481,-0.283559,-0.283869,-0.283976,-0.284060,-0.283877,-0.284000,-0.283998,-0.283292,-0.284150
1,-0.011577,-0.011672,-0.011886,-0.011754,-0.011719,-0.011904,-0.011460,-0.011801,-0.011679,-0.012007,-0.012006,-0.011633,-0.011853,-0.011629,-0.011534,-0.011440,-0.011602,-0.011407,-0.011951,-0.011236
2,-1.597049,-1.597076,-1.596634,-1.597453,-1.596846,-1.596406,-1.596944,-1.597226,-1.597204,-1.596918,-1.596598,-1.597133,-1.596860,-1.596446,-1.596483,-1.597202,-1.597593,-1.596804,-1.597156,-1.597035
3,-1.211432,-1.210818,-1.211399,-1.211176,-1.211758,-1.211650,-1.211330,-1.211436,-1.211300,-1.211313,-1.211382,-1.211760,-1.211371,-1.211670,-1.211864,-1.211534,-1.211325,-1.210681,-1.211282,-1.211143
4,1.742279,1.742013,1.742542,1.742156,1.742604,1.741968,1.741927,1.742983,1.741764,1.741728,1.742488,1.741976,1.742525,1.742073,1.742370,1.742472,1.741824,1.741895,1.741383,1.741869
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,1.726028,1.725907,1.725471,1.725809,1.726493,1.725928,1.725585,1.726317,1.725654,1.725657,1.725922,1.725892,1.726021,1.726332,1.726587,1.725540,1.726236,1.725329,1.725697,1.726001
296,-0.305347,-0.305079,-0.304981,-0.305953,-0.304916,-0.305238,-0.305834,-0.305528,-0.304937,-0.305592,-0.305609,-0.305918,-0.305430,-0.305958,-0.305253,-0.305631,-0.306057,-0.305100,-0.304653,-0.305110
297,1.329103,1.329072,1.329580,1.328805,1.329251,1.329374,1.328719,1.328448,1.329461,1.329803,1.328950,1.329539,1.329548,1.328953,1.329232,1.329450,1.329074,1.328743,1.328624,1.328874
298,0.549281,0.548557,0.548998,0.549691,0.548811,0.549254,0.549299,0.549611,0.549413,0.549865,0.549192,0.549919,0.549382,0.549147,0.548674,0.550105,0.549202,0.549758,0.549040,0.549139


### **Градиентный спуск для Lasso регрессии.**

In [36]:
def LassoGrad(X, Y, w, lmbd, lr):
    X = X.to_numpy()
    #текущее предсказание
    pred = X @ w #вектор
    #ошибка на каждом объекте
    error = pred - Y #вектор
    #распространим ошибку объекта на все его признаки
    grad_for_every_feature = 2 * X * error[:, None] #матрица
    #вычислим средний градиент веса
    grad = np.mean(grad_for_every_feature, axis=0)
    #добавим к градиенту штраф за размер веса
    grad += lmbd * np.sign(w)
    #изменим веса
    w = w - grad * lr
    return w

### **Напишем свою функцию fit для Лассо-регрессии.**

In [37]:
def fit(X, Y):
    #создадим изначальные веса
    w = np.zeros(X.shape[1])
    #разница предыдущей и текущей ошибки
    dif = 10
    while dif > 1e-6:
        #предыдущая ошибка
        last_error = np.mean((X @ w - Y) ** 2)
        #сделаем один шаг спуска
        w = LassoGrad(X, Y, w, 1e-3, 1e-5)
        #узнаем текущую ошибку
        cur_error = np.mean((X @ w - Y) ** 2)
        #обновим разницу ошибки
        dif = abs(last_error - cur_error)
    return w

In [41]:
new_w = fit(X_std, Y)
#узнаем её ошибку
pred = (X_std @ new_w)
error = np.mean((pred - Y)**2)
error

np.float64(58372.241769004315)